# MIA Visualization

In [1]:
import os
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

In [2]:
# MIA and DCR vs. Train Steps
train_steps_dcr = [0.66060, 0.67230, 0.68049, 0.67850, 0.70130, 0.71829, 0.74869, 0.76339, 0.77380, 0.77319]
train_steps_wb = [0.09900, 0.12400, 0.13600, 0.17300, 0.26000, 0.37300, 0.42500, 0.48000, 0.49000, 0.51500]
train_steps_bb = [0.10900, 0.11800, 0.11100, 0.10900, 0.11300, 0.15300, 0.20000, 0.24700, 0.28400, 0.27800]
train_steps = [5e03, 1e04, 1.5e04, 2.5e04, 5e04, 1e05, 2e05, 3e05, 4e05, 5e05]

# MIA and DCR vs. Diffusion steps
diffusion_steps_dcr = [0.71149, 0.73510, 0.76400, 0.77149, 0.70600, 0.71430, 0.71000, 0.74450, 0.70990, 0.70969]
diffusion_steps_wb = [0.18900, 0.24300, 0.25600, 0.26700, 0.44200, 0.45500, 0.44000, 0.44500, 0.43700, 0.42600]
diffusion_steps_bb = [0.13000, 0.18600, 0.20100, 0.22700, 0.26000, 0.26100, 0.27100, 0.25200, 0.25900, 0.25500]
diffusion_steps = [10, 20, 50, 80, 100, 500, 1000, 2000, 3000, 4000]

# MIA and DCR vs. Synthetic data
synthetic_size = ["1x", "10x", "1M"]
bb_10k = [0.34, 0.48, 0.48]
bb_20k = [0.24, 0.33, 0.35]
bb_50k = [0.11, 0.14, 0.14]
bb_100k = [0.1, 0.112, 0.112]
dcr_20k = [0.749, 0.748, 0.749]

# MIA and DCR vs. Batch Size
batch_size_dcr = [0.7490, 0.7445, 0.7922]
batch_size_wb = [0.4390, 0.4450, 0.484]
batch_size_bb = [0.2450, 0.2520, 0.2550]
batch_size = [2048, 4096, 8192]

This notebook will visualize how each of the following variable affects the fairness towards a particular protected group.

In [3]:
INDEPENDENT_VARIABLES = ["model", "dataset", "num_params"]
FIG_HEIGHT = 1050
FIG_WIDTH = 1500
FONT_SIZE = 24
# Select from "grouped_race", "grouped_gender", "grouped_sexuality"
SENSITIVE_ATTRIBUTE = "grouped_gender"

## Utilities

In [4]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [5]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/"
plot_name = "mia_dcr_results"

In [6]:
def customize_figure_layout(gen_fig: Any) -> None:

    gen_fig.update_layout(
        height=FIG_HEIGHT,
        width=FIG_WIDTH,
        font_color="black",
        legend=dict(orientation="h"),
        title_font_family="Helvetica",
        title_x=0.47,
        margin=dict(l=30, r=30, t=50, b=30),
        plot_bgcolor="#eeeeee",
        font=dict(size=FONT_SIZE, family="Helvetica"),
    )


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{plot_name}.png"
    plot_pdf_path = f"{plots_dir}/{plot_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)

    display(gen_fig)

In [7]:
gen_fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=False,
    row_heights=[1000, 1000],
    vertical_spacing=0.1,
    horizontal_spacing=0.21,
    subplot_titles=["MIA and DCR vs. Training Steps", "MIA and DCR vs. Diffusion Steps", "MIA and DCR vs. Synthetic Generation Size", "MIA and DCR vs. Batch Size"]
)

for annotation in gen_fig["layout"]["annotations"]:
    annotation["font"] = dict(size=24, family="Helvetica")  # Setting size to 20 and font family to Courier

# Train steps figure
fig = go.Scatter(
    x=train_steps,
    y=train_steps_wb,
    name = "MIA (WB)",
    legendgroup = '1',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=1)

fig = go.Scatter(
    x=train_steps,
    y=train_steps_bb,
    name="MIA (BB)",
    legendgroup = '1',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=1)

fig = go.Scatter(
    x=train_steps,
    y=train_steps_dcr,
    name="DCR",
    legendgroup = '1',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=1)
legend_name = "legend2"
gen_fig.update_traces(row=1, col=1, legend=legend_name)
gen_fig.update_layout({legend_name: dict(orientation="h", x=0.4, y=1, bgcolor='rgba(0,0,0,0)')})

# Diffusion steps figure
fig = go.Scatter(
    x=diffusion_steps,
    y=diffusion_steps_wb,
    name="MIA (WB)",
    legendgroup = '2',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=2)

fig = go.Scatter(
    x=diffusion_steps,
    y=diffusion_steps_bb,
    name = "MIA (BB)",
    legendgroup = '2',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=2)

fig = go.Scatter(
    x=diffusion_steps,
    y=diffusion_steps_dcr,
    name = "DCR",
    legendgroup = '2',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=1, col=2)
legend_name = "legend3"
gen_fig.update_traces(row=1, col=2, legend=legend_name)
gen_fig.update_layout({legend_name: dict(orientation="h", x=1.01, y=1, bgcolor='rgba(0,0,0,0)')})

# Synthetic size figure

fig = go.Scatter(
    x=synthetic_size,
    y=bb_10k,
    name = "MIA (BB) 10K",
    legendgroup = '3',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=1)

fig = go.Scatter(
    x=synthetic_size,
    y=bb_20k,
    name = "MIA (BB) 20K",
    legendgroup = '3',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=1)

fig = go.Scatter(
    x=synthetic_size,
    y=bb_50k,
    name = "MIA (BB) 50K",
    legendgroup = '3',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=1)

fig = go.Scatter(
    x=synthetic_size,
    y=bb_100k,
    name = "MIA (BB) 100K",
    legendgroup = '3',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=1)

fig = go.Scatter(
    x=synthetic_size,
    y=dcr_20k,
    name = "DCR 20K",
    legendgroup = '3',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=1)
legend_name = "legend4"
gen_fig.update_traces(row=2, col=1, legend=legend_name)
gen_fig.update_layout({legend_name: dict(orientation="h", x=0.4, y=0.275, bgcolor='rgba(0,0,0,0)')})

# Batch Size figure

fig = go.Scatter(
    x=batch_size,
    y=batch_size_wb,
    name = "MIA (WB)",
    legendgroup = '4',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=2)

fig = go.Scatter(
    x=batch_size,
    y=batch_size_bb,
    name = "MIA (BB)",
    legendgroup = '4',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=2)

fig = go.Scatter(
    x=batch_size,
    y=batch_size_dcr,
    name = "DCR",
    legendgroup = '4',
    marker=dict(size=10),
	line=dict(width=4),
    showlegend=True,
)

gen_fig.append_trace(fig, row=2, col=2)
legend_name = "legend5"
gen_fig.update_traces(row=2, col=2, legend=legend_name)
gen_fig.update_layout({legend_name: dict(orientation="h", x=1.01, y=0.4, bgcolor='rgba(0,0,0,0)')})

gen_fig.update_xaxes(
    matches=None, tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE
)
gen_fig.update_xaxes(type="log", tickformat=".1E", tickangle=-25, row=1, col=1)
gen_fig.update_xaxes(type="log", tickformat=".1E", tickangle=-25, row=1, col=2)

gen_fig.update_xaxes(type='category', row=2, col=2)

customize_figure_layout(gen_fig)
save_and_display_figure(gen_fig, plot_name, plots_dir)